In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, roc_auc_score
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import os

def magnitude_norm(data, data_min=0):
    data_max = np.max(np.abs(data))
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

In [2]:
info_df = pd.read_csv("../data/input_info_VRC01_IC80.csv")
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
N_SEQ, L_SEQ = aligned_sequences.shape
seq_no_array = info_df['Seq_no'].values
matches = aligned_sequences[:, None, :] == aligned_sequences[None, :, :]

seq_sim_mat = (aligned_sequences[:, None, :] == aligned_sequences[None, :, :]).mean(axis=2)
max_sim = (seq_sim_mat - np.eye(N_SEQ)).max(axis=1)

mean_max = np.mean(max_sim)
std_max = np.std(max_sim)
print("Mean-Max Sequence Idenity in the full dataset =", mean_max, "+/-", std_max)

Mean-Max Sequence Idenity in the full dataset = 0.9012491859570184 +/- 0.07425929352624758


In [3]:
S  = aligned_sequences.copy()
N  = len(S)

# identity = fraction of matching alignment columns, gaps as characters
sim = (S[:, None, :] == S[None, :, :]).mean(axis=2)

nnb = (sim - np.eye(N)).max(axis=1)
print(f"thresh\t| clust\t| N_e\t| N_e/N\t| largest")
for t in (0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99):
    A       = sim >= t
    n_eff   = (1.0 / A.sum(axis=1)).sum()
    k, lab  = connected_components(csr_matrix(A), directed=False)
    print(f"{t}\t| {k}\t| {round(n_eff, 1)}\t| {round(n_eff/N, 3)}\t| {np.bincount(lab).max()}")

thresh	| clust	| N_e	| N_e/N	| largest
0.7	| 1	| 1.5	| 0.002	| 889
0.75	| 2	| 12.7	| 0.014	| 888
0.8	| 67	| 114.8	| 0.129	| 788
0.85	| 385	| 445.8	| 0.501	| 195
0.9	| 630	| 632.1	| 0.711	| 12
0.95	| 656	| 659.8	| 0.742	| 12
0.99	| 746	| 754.3	| 0.849	| 11


In [4]:
root_dir = "/pl/active/rdi_data/fahsai/PLM"
pred_dir = os.path.join(root_dir, "results/predictions/classification")
attr_dir = os.path.join(root_dir, "results/attribution_maps/classification")
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]

# for sim_cutoff in [0.80, 0.85, 0.90, 0.95, 0.99, 1.01]:
#     mask = max_sim < sim_cutoff
edges = [0, 0.80, 0.85, 0.90, 0.99, 1.001]
for lo, hi in zip(edges, edges[1:]):
    mask = (max_sim >= lo) & (max_sim < hi)
    print(f"Processing sequences with {lo}-{hi}, n={mask.sum()}")
    
    model_metrics_records = []
    attr_metrics_records = []

    for model_idx, model_id in enumerate(selected_models):
        predict_df = pd.read_csv(os.path.join(pred_dir, f"train_{model_id}.csv"))
        assert (predict_df["Seq_no"] == seq_no_array).all()
        true_labels = predict_df["Label"].values
        pred_labels = []
        pos_pred_probabilities = []
        
        for pred in predict_df["Prediction"]:
            pred_0 = float(pred.split()[1].split(",")[0])
            pred_1 = float(pred.split()[3].split("}")[0])
            pos_pred_probabilities.append(pred_1)
            pred_labels.append(0 if pred_0 > pred_1 else 1)

        pred_labels = np.array(pred_labels)
        assert len(true_labels) == len(pred_labels)

        model_metrics_records.append({
            "Model_ID": model_id,
            "N": mask.sum(),
            "AUC": roc_auc_score(true_labels[mask], np.array(pos_pred_probabilities)[mask]),
            "balanced-Acc": accuracy_score(true_labels[mask], pred_labels[mask]),
            "F1": f1_score(true_labels[mask], pred_labels[mask]),
        })

        all_attr_array_0 = np.zeros([N_SEQ, L_SEQ])
        all_attr_array_1 = np.zeros([N_SEQ, L_SEQ])

        for seq_idx, no in enumerate(info_df['Seq_no']):
            attr_0 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_0.npy"))
            all_attr_array_0[seq_idx] = magnitude_norm(attr_0)

            attr_1 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_1.npy"))
            all_attr_array_1[seq_idx] = magnitude_norm(attr_1)

        # Resolve Attribution Conflicts
        attr_1_array = all_attr_array_1.copy()
        attr_0_array = all_attr_array_0.copy()

        pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
        neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
        conflict_mask = (pos_conflict_mask | neg_conflict_mask)
        align_mask = ~conflict_mask

        abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
        abs_attr_score_array[conflict_mask] = 0
        attr_score_array = abs_attr_score_array.copy()
        attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

        attr_pred = attr_score_array.sum(axis=1)
        attr_pred[attr_pred < 0] = 0
        attr_pred[attr_pred > 0] = 1

        attr_metrics_records.append({
            "Model_ID": model_id,
            "AUC": roc_auc_score(true_labels[mask], attr_score_array.sum(axis=1)[mask]),
            "balanced-Acc": accuracy_score(true_labels[mask], attr_pred[mask]),
            "F1": f1_score(true_labels[mask], attr_pred[mask])
        })


    df = pd.DataFrame(model_metrics_records)
    df["similarity-cutoff"] = f"{lo}-{hi}"
    print("--- Model Prediction ---")
    for score in ["AUC", "balanced-Acc", "F1"]:
        mean_score = df[score].mean()
        std_score = df[score].std()
        print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

    attr_df = pd.DataFrame(attr_metrics_records)
    attr_df["similarity-cutoff"] = f"{lo}-{hi}"
    print("--- Attribution Prediction ---")
    for score in ["AUC", "balanced-Acc", "F1"]:
        mean_score = attr_df[score].mean()
        std_score = attr_df[score].std()
        print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

    print("-----------\n")

Processing sequences with 0-0.8, n=46
--- Model Prediction ---
AUC = 0.987 +/- 0.023
balanced-Acc = 0.983 +/- 0.018
F1 = 0.957 +/- 0.046
--- Attribution Prediction ---
AUC = 0.968 +/- 0.045
balanced-Acc = 0.987 +/- 0.019
F1 = 0.967 +/- 0.049
-----------

Processing sequences with 0.8-0.85, n=249
--- Model Prediction ---
AUC = 0.986 +/- 0.010
balanced-Acc = 0.972 +/- 0.010
F1 = 0.946 +/- 0.020
--- Attribution Prediction ---
AUC = 0.982 +/- 0.012
balanced-Acc = 0.968 +/- 0.009
F1 = 0.939 +/- 0.019
-----------

Processing sequences with 0.85-0.9, n=217
--- Model Prediction ---
AUC = 0.978 +/- 0.009
balanced-Acc = 0.929 +/- 0.012
F1 = 0.898 +/- 0.017
--- Attribution Prediction ---
AUC = 0.958 +/- 0.015
balanced-Acc = 0.931 +/- 0.010
F1 = 0.902 +/- 0.016
-----------

Processing sequences with 0.9-0.99, n=147
--- Model Prediction ---
AUC = 0.971 +/- 0.009
balanced-Acc = 0.916 +/- 0.014
F1 = 0.884 +/- 0.020
--- Attribution Prediction ---
AUC = 0.952 +/- 0.014
balanced-Acc = 0.914 +/- 0.016
F1

In [5]:
result_dir = "/pl/active/rdi_data/fahsai/PLM"
predict_df = pd.read_csv(result_dir+"/results/cv/predictions_summary_classification.csv")
pred_labels = []
for pred in predict_df["prediction"]:
    pred_0 = float(pred.split()[1].split(",")[0])
    pred_1 = float(pred.split()[3].split("}")[0])
    pred_labels.append(0 if pred_0 > pred_1 else 1)
predict_df["predicted_label"] = pred_labels

In [6]:
cutoff = 95

for n_fold in range(1,6):
    print("Processing fold -", n_fold)
    fold_df = predict_df[predict_df["fold"] == n_fold]
    print("-- Full dataset --")
    print("Accuracy:", accuracy_score(fold_df["actual_label"], fold_df["predicted_label"]))
    print("F1-score:", f1_score(fold_df["actual_label"], fold_df["predicted_label"]))
    
    train_fold_df = fold_df[fold_df["split"] == "Train"]
    train_fold_seqno = train_fold_df["Seq_no"].values
    train_idx = np.where(np.isin(seq_no_array, train_fold_seqno))[0]
    print(f"-- Train set (n={len(train_fold_seqno)}) --")
    print("Accuracy:", accuracy_score(train_fold_df["actual_label"], train_fold_df["predicted_label"]))
    print("F1-score:", f1_score(train_fold_df["actual_label"], train_fold_df["predicted_label"]))
    
    test_fold_df = fold_df[fold_df["split"] == "Test"]
    test_fold_seqno = test_fold_df["Seq_no"].values
    test_idx = np.where(np.isin(seq_no_array, test_fold_seqno))[0]
    print(f"-- Test set (n={len(test_fold_seqno)}) --")
    print("Accuracy:", accuracy_score(test_fold_df["actual_label"], test_fold_df["predicted_label"]))
    print("F1-score:", f1_score(test_fold_df["actual_label"], test_fold_df["predicted_label"]))

    sim_block = seq_sim_mat[np.ix_(test_idx, train_idx)] * 100
    n_test_in_train = (np.max(sim_block, axis=1) > cutoff).sum()
    overlap = n_test_in_train / len(test_idx)
    print(f"Train-test overlap (>{cutoff}% sim):", overlap, f"({n_test_in_train}/{len(test_idx)})")
    print("-------------------")

Processing fold - 1
-- Full dataset --
Accuracy: 0.8627671541057368
F1-score: 0.7910958904109588
-- Train set (n=711) --
Accuracy: 0.8945147679324894
F1-score: 0.8400852878464818
-- Test set (n=178) --
Accuracy: 0.7359550561797753
F1-score: 0.591304347826087
Train-test overlap (>95% sim): 0.398876404494382 (71/178)
-------------------
Processing fold - 2
-- Full dataset --
Accuracy: 0.8526434195725534
F1-score: 0.7805695142378559
-- Train set (n=711) --
Accuracy: 0.8762306610407876
F1-score: 0.8166666666666667
-- Test set (n=178) --
Accuracy: 0.7584269662921348
F1-score: 0.6324786324786325
Train-test overlap (>95% sim): 0.37640449438202245 (67/178)
-------------------
Processing fold - 3
-- Full dataset --
Accuracy: 0.8785151856017998
F1-score: 0.8235294117647058
-- Train set (n=711) --
Accuracy: 0.9184247538677919
F1-score: 0.8796680497925312
-- Test set (n=178) --
Accuracy: 0.7191011235955056
F1-score: 0.6153846153846154
Train-test overlap (>95% sim): 0.29213483146067415 (52/178)
---

In [7]:
N_MODEL = len(selected_models)

all_attr_array_0 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
all_attr_array_1 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
attn_scalar_array = np.zeros((N_MODEL, N_SEQ, L_SEQ, L_SEQ))
cls_attn_scalar_array =  np.zeros([N_MODEL, N_SEQ, L_SEQ])

for n, model_name in enumerate(selected_models):
    for i, no in enumerate(seq_no_array):
        attr_0 = np.load(os.path.join(root_dir, attr_dir, model_name, str(no)+"_attribution_ig_class_0.npy"))
        all_attr_array_0[n,i] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(root_dir, attr_dir, model_name, str(no)+"_attribution_ig_class_1.npy"))
        all_attr_array_1[n,i] = magnitude_norm(attr_1)

attr_1_array = all_attr_array_1.copy()
attr_0_array = all_attr_array_0.copy()

pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
conflict_mask = (pos_conflict_mask | neg_conflict_mask)
align_mask = ~conflict_mask

abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
abs_attr_score_array[conflict_mask] = 0
attr_score_array = abs_attr_score_array.copy()
# assign signs by class 1 attribution from each model
attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

In [8]:
A = np.mean(attr_score_array, axis=0) # A[s, i] = attribution of residue i in sequence s
S = aligned_sequences.copy()

rows = []
for i in range(L_SEQ):
    for aa in np.unique(S[:, i]):
        m = S[:, i] == aa
        if m.sum() >= 5:
            rows.append((i, aa, m.sum(), A[m, i].mean(), A[m, i].var()))
d = pd.DataFrame(rows, columns=["pos","aa","n","mean","var"])

within  = (d["n"] * d["var"]).sum() / d["n"].sum()          # context-dependent part
between = np.cov(d["mean"], aweights=d["n"])                # what a per-position table could capture
print(f"context-dependent fraction of attribution variance = {within/(within+between):.3f}")

context-dependent fraction of attribution variance = 0.338
